# 第二次作业 03 固定解析版本与断点续采

保存固定解析器的续采和覆盖检查。原采集窗口限制保留，不修改日期或绕过截止条件。

题目入口、固定实验来源和环境说明见 [第二次作业说明](railway_analysis/README.md)。

先运行本文件的**公共初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

**历史采集流程**：2026年11月的原采集窗口已结束。请结合已保存结果阅读，勿将“全部运行”视为重新采集入口；代码中的日期、截止条件和来源区分原样保留。

[上一份 02](hw02_02_direct_collection.ipynb) · [下一份 04](hw02_04_source_audit_and_routes.ipynb)
脚本与已有数据现已迁入本仓库的 `railway_analysis/`。下方历史输出保留原路径和运行时间；本次仅整理结构，没有重新采集或拟合模型。


## 公共初始化

定位本仓库中的作业目录，检查环境和固定代码版本；不运行采集或模型训练。


In [ ]:
# 标准库
import importlib
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 与第一次作业相同：定位专题目录，再使用公共初始化。
# 支持从仓库根目录或 notebooks/ 启动Jupyter，不限定安装盘符。
search_start = Path.cwd().resolve()
for candidate in (search_start, *search_start.parents):
    notebook_dir = candidate if (candidate / "railway_analysis/common.py").is_file() else candidate / "notebooks"
    if (notebook_dir / "railway_analysis/common.py").is_file():
        if str(notebook_dir) not in sys.path:
            sys.path.insert(0, str(notebook_dir))
        break
else:
    raise FileNotFoundError("未找到 railway_analysis/common.py，请从仓库或 notebooks 目录启动Jupyter。")

from railway_analysis.common import initialize_project

REQUIRED_LOCAL_MODULES = (
    'stage1_collect',
    'stage1c_collect_v2',
    'stage1c_parser_v2',
    'stage1c_resume_v2',
)
ROOT = initialize_project(REQUIRED_LOCAL_MODULES)
SCRIPTS_DIR = ROOT / 'scripts'


## 问题1—核心代码8：固定解析版本的断点续采

In [1]:
# 功能：在统一环境初始化后加载业务模块，续采余下320组；每组保存进度，统一解析和导出。

# 注意：查询日仍是11月18日。原快照跨度上限48小时从11月11日21:28计起，
#       因而请在11月13日21:28前完成本次续采；若过期，回传提示，不改时间戳绕过限制。

import stage1_collect
import stage1c_parser_v2
import stage1c_collect_v2
import stage1c_resume_v2
for module in (stage1_collect, stage1c_parser_v2, stage1c_collect_v2, stage1c_resume_v2):
    importlib.reload(module)

MAX_QUERIES = 995  # 实际只请求剩余部分；可改100先采一批，再运行本段续采
collection = stage1c_resume_v2.collect(max_queries=MAX_QUERIES)
COLLECTION_OUT = Path(collection["output_dir"])
for key in ["query_date", "parser_version", "completed_queries", "remaining_queries", "train_records",
            "unavailable_timetable_records", "unresolved_station_record_count", "blocking_error"]:
    print(f"{key}: {collection[key]}")
print("保存位置：", COLLECTION_OUT)
print("matrix_ready:", collection["matrix_ready"], "（直达查询表还需中转及站点覆盖核验）")


采集目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\raw\user_stage1c_v2_20261113_132207_325427
676/995 重庆→昆明: validated_records; valid=22
677/995 重庆→拉萨: validated_records; valid=1
678/995 重庆→西安: validated_records; valid=39
679/995 重庆→兰州: validated_records; valid=8
700/995 成都→武汉: validated_records; valid=33
725/995 贵阳→杭州: validated_records; valid=24
750/995 昆明→呼和浩特: validated_records; valid=1
775/995 昆明→乌鲁木齐: validated_records; valid=1
800/995 拉萨→贵阳: no_direct_train_returned_not_unreachable; valid=0
825/995 西安→长沙: partial_timetable_unavailable; valid=25
850/995 兰州→合肥: validated_records; valid=13
875/995 西宁→沈阳: validated_records; valid=2
900/995 银川→北京: validated_records; valid=11
925/995 银川→拉萨: no_direct_train_returned_not_unreachable; valid=0
950/995 乌鲁木齐→南宁: no_direct_train_returned_not_unreachable; valid=0
975/995 香港→南昌: validated_records; valid=2
995/995 珠海→珠海: validated_records; valid=99
query_date: 2026-11-18
parser_version: stage1c-station-metadata-v2
completed_queries: 995
remain

## 问题1—核心代码9：覆盖与缺失检查

前置变量或模块：`COLLECTION_OUT`、`collection`。

In [2]:
# 功能：汇总全部995组查询状态、检查34节点城市对表，并统计隔离的站码记录。
# 说明：空直达响应、时刻未知、站码未核实均按各自状态保留，不视作真实不可达。
quality = pd.read_csv(COLLECTION_OUT / "query_quality.csv", encoding="utf-8-sig")
direct = pd.read_csv(COLLECTION_OUT / "direct_screen_34x34_long.csv", encoding="utf-8-sig")
unresolved = pd.read_csv(COLLECTION_OUT / "unresolved_station_records.csv", encoding="utf-8-sig")
assert len(direct) == 34 * 34
assert direct.loc[direct["from_node"] == direct["to_node"], "observed_direct_min_minutes"].eq(0).all()
query_counts = quality["status"].value_counts(dropna=False).rename_axis("query_status").to_frame("count")
pair_counts = direct["status"].value_counts(dropna=False).rename_axis("pair_status").to_frame("count")
display(query_counts)
display(pair_counts)
display(unresolved[["query_id", "raw_row_index"]])
query_counts.to_csv(COLLECTION_OUT / "query_status_counts.csv", encoding="utf-8-sig")
pair_counts.to_csv(COLLECTION_OUT / "pair_status_counts.csv", encoding="utf-8-sig")
print("剩余请求：", collection["remaining_queries"], "；站码待核验记录：", len(unresolved))


,count
query_status,
validated_records,793
no_direct_train_returned_not_unreachable,134
partial_timetable_unavailable,63
partial_station_metadata_unresolved,5


,count
pair_status,
observed_direct_min_only_not_shortest_itinerary,788
no_direct_record_returned_not_unreachable,134
special_region_connection_review_pending,130
observed_direct_min_only_not_shortest_itinerary_with_unknown_timetables,47
diagonal,34
unknown_timetable_no_accepted_known_duration,16
observed_direct_min_only_not_shortest_itinerary_with_unresolved_station_metadata,5
returned_stations_outside_draft_set,2


,query_id,raw_row_index
0,N07_N06,27
1,N07_N06,120
2,N22_N24,52
3,N22_N24,107
4,N22_N24,114
5,N23_N22,76
6,N23_N22,82
7,N23_N22,133
8,N23_N24,80
9,N24_N22,89


剩余请求： 0 ；站码待核验记录： 10
